<a href="https://colab.research.google.com/github/anupams091/NFL-Prediction-Model/blob/main/NFLGame_predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q nflreadpy streamlit

import nflreadpy as nfl
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50)

print("nflreadpy version:", nfl.__version__)
print("pandas version:", pd.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 24.4 MB/s eta 0:00:00
nflreadpy version: 0.1.5
pandas version: 2.2.3


In [2]:
seasons = list(range(2003, 2027))

schedules = nfl.load_schedules(seasons).to_pandas()

print("Rows, Columns:", schedules.shape)

schedules[["game_id", "season", "game_type", "week", "gameday",
           "away_team", "away_score", "home_team", "home_score",
           "result", "spread_line"]].tail(10)

Rows, Columns: (6504, 46)


,game_id,season,game_type,week,gameday,away_team,away_score,home_team,home_score,result,spread_line
6494,2026_18_DET_GB,2026,REG,18,2027-01-10,DET,NaN,GB,NaN,NaN,NaN
6495,2026_18_TEN_HOU,2026,REG,18,2027-01-10,TEN,NaN,HOU,NaN,NaN,NaN
6496,2026_18_JAX_IND,2026,REG,18,2027-01-10,JAX,NaN,IND,NaN,NaN,NaN
6497,2026_18_LV_KC,2026,REG,18,2027-01-10,LV,NaN,KC,NaN,NaN,NaN
6498,2026_18_SEA_LA,2026,REG,18,2027-01-10,SEA,NaN,LA,NaN,NaN,NaN
6499,2026_18_CHI_MIN,2026,REG,18,2027-01-10,CHI,NaN,MIN,NaN,NaN,NaN
6500,2026_18_MIA_NE,2026,REG,18,2027-01-10,MIA,NaN,NE,NaN,NaN,NaN
6501,2026_18_TB_NO,2026,REG,18,2027-01-10,TB,NaN,NO,NaN,NaN,NaN
6502,2026_18_PHI_NYG,2026,REG,18,2027-01-10,PHI,NaN,NYG,NaN,NaN,NaN
6503,2026_18_DAL_WAS,2026,REG,18,2027-01-10,DAL,NaN,WAS,NaN,NaN,NaN


In [3]:
played = schedules[schedules["result"].notna()]
upcoming = schedules[schedules["result"].isna()]

print("Games played:", len(played))
print("Games not played yet:", len(upcoming))
print()
print("Game types in the data:")
print(schedules["game_type"].value_counts())

Games played: 6296
Games not played yet: 208

Game types in the data:
game_type
REG    6239
WC      104
DIV      92
CON      46
SB       23
Name: count, dtype: int64


In [4]:
played = played.copy()
played["home_win"] = (played["result"] > 0).astype(int)

home_baseline = played["home_win"].mean()

has_favorite = played[played["spread_line"] != 0]
vegas_correct = (has_favorite["spread_line"] > 0) == (has_favorite["result"] > 0)
vegas_baseline = vegas_correct.mean()

print(f"Always pick the home team:      {home_baseline:.1%}")
print(f"Always pick the Vegas favorite: {vegas_baseline:.1%}")

played.groupby("season")["home_win"].mean().round(3)

Always pick the home team:      56.1%
Always pick the Vegas favorite: 66.7%


,home_win
season,
2003,0.614
2004,0.566
2005,0.584
2006,0.539
2007,0.569
2008,0.566
2009,0.573
2010,0.554
2011,0.573


In [5]:
relocations = {"OAK": "LV", "SD": "LAC", "STL": "LA"}

schedules["home_team"] = schedules["home_team"].replace(relocations)
schedules["away_team"] = schedules["away_team"].replace(relocations)

print("Unique teams after fix:", schedules["home_team"].nunique())

played = schedules[schedules["result"].notna()].copy()
played["home_win"] = (played["result"] > 0).astype(int)
upcoming = schedules[schedules["result"].isna()].copy()

print("Played:", len(played), "| Upcoming:", len(upcoming))

Unique teams after fix: 32
Played: 6296 | Upcoming: 208


In [6]:
cols = ["game_id", "season", "week", "gameday"]

home = played[cols + ["home_team", "away_team", "home_score", "away_score"]].rename(columns={
    "home_team": "team", "away_team": "opponent",
    "home_score": "points_for", "away_score": "points_against"})
home["is_home"] = 1

away = played[cols + ["away_team", "home_team", "away_score", "home_score"]].rename(columns={
    "away_team": "team", "home_team": "opponent",
    "away_score": "points_for", "home_score": "points_against"})
away["is_home"] = 0

team_games = pd.concat([home, away], ignore_index=True)
team_games["gameday"] = pd.to_datetime(team_games["gameday"])
team_games["point_diff"] = team_games["points_for"] - team_games["points_against"]
team_games["win"] = (team_games["point_diff"] > 0).astype(int)

team_games = team_games.sort_values(["team", "gameday"]).reset_index(drop=True)

print(team_games.shape)
team_games[team_games["team"] == "KC"].tail(5)

(12592, 11)


,game_id,season,week,gameday,team,opponent,points_for,points_against,is_home,point_diff,win
6279,2025_18_KC_LV,2025,18,2026-01-04,KC,LV,12.0,14.0,0,-2.0,0
6280,2026_01_DEN_KC,2026,1,2026-09-14,KC,DEN,31.0,10.0,1,21.0,1
6281,2026_02_IND_KC,2026,2,2026-09-20,KC,IND,33.0,30.0,1,3.0,1
6282,2026_03_KC_MIA,2026,3,2026-09-27,KC,MIA,24.0,10.0,0,14.0,1
6283,2026_04_KC_LV,2026,4,2026-10-04,KC,LV,30.0,27.0,0,3.0,1


In [7]:
WINDOW = 8

for stat in ["point_diff", "points_for", "points_against", "win"]:
    team_games[f"{stat}_last{WINDOW}"] = (
        team_games.groupby("team")[stat]
        .transform(lambda s: s.shift(1).rolling(WINDOW, min_periods=1).mean())
    )

print(team_games.shape)
team_games[team_games["team"] == "KC"][
    ["game_id", "point_diff", "point_diff_last8", "win", "win_last8"]
].tail(10).round(2)

(12592, 15)


,game_id,point_diff,point_diff_last8,win,win_last8
6274,2025_13_KC_DAL,-3.0,9.00,0,0.62
6275,2025_14_HOU_KC,-10.0,6.50,0,0.50
6276,2025_15_LAC_KC,-3.0,5.62,0,0.50
6277,2025_16_KC_TEN,-17.0,3.62,0,0.38
6278,2025_17_DEN_KC,-7.0,-2.38,0,0.25
6279,2025_18_KC_LV,-2.0,-5.88,0,0.12
6280,2026_01_DEN_KC,21.0,-5.25,1,0.12
6281,2026_02_IND_KC,3.0,-2.25,1,0.25
6282,2026_03_KC_MIA,14.0,-2.25,1,0.25
6283,2026_04_KC_LV,3.0,-0.12,1,0.38


In [8]:
BASE = 1500      # starting rating for every team
K = 20           # how fast ratings react to each game
HFA = 48         # home field advantage, in Elo points
REVERT = 1/3     # how far ratings move back to 1500 each offseason

played["gameday"] = pd.to_datetime(played["gameday"])
played = played.sort_values(["gameday", "game_id"]).reset_index(drop=True)

elo = {}
current_season = None
home_pre, away_pre = [], []

for row in played.itertuples():
    if row.season != current_season:
        elo = {team: r + (BASE - r) * REVERT for team, r in elo.items()}
        current_season = row.season

    h = elo.get(row.home_team, BASE)
    a = elo.get(row.away_team, BASE)
    home_pre.append(h)
    away_pre.append(a)

    exp_home = 1 / (1 + 10 ** (-(h + HFA - a) / 400))
    actual = 1.0 if row.result > 0 else 0.0 if row.result < 0 else 0.5

    winner_diff = (h + HFA - a) if row.result > 0 else (a - h - HFA)
    mov_mult = np.log(abs(row.result) + 1) * 2.2 / (winner_diff * 0.001 + 2.2)
    change = K * mov_mult * (actual - exp_home)

    elo[row.home_team] = h + change
    elo[row.away_team] = a - change

played["home_elo"] = home_pre
played["away_elo"] = away_pre
played["elo_prob"] = 1 / (1 + 10 ** (-(played["home_elo"] + HFA - played["away_elo"]) / 400))

current_elo = pd.Series(elo).sort_values(ascending=False).round(0)
print("Top 5 right now:\n", current_elo.head(5))
print("\nBottom 3 right now:\n", current_elo.tail(3))

Top 5 right now:
 SEA    1683.0
SF     1634.0
BUF    1626.0
MIN    1614.0
JAX    1612.0
dtype: float64

Bottom 3 right now:
 MIA    1384.0
NYJ    1362.0
TEN    1303.0
dtype: float64


In [9]:
played["elo_pick"] = (played["elo_prob"] > 0.5).astype(int)
played["elo_correct"] = (played["elo_pick"] == played["home_win"]).astype(int)

eval_games = played[played["season"] >= 2005]

accuracy = eval_games["elo_correct"].mean()
brier = ((eval_games["elo_prob"] - eval_games["home_win"]) ** 2).mean()

print(f"Elo accuracy (2005 onward): {accuracy:.1%}")
print(f"Elo Brier score:            {brier:.3f}")
print(f"Coin flip Brier score:      {0.25:.3f}")

eval_games.groupby("season")["elo_correct"].mean().round(3).tail(8)

Elo accuracy (2005 onward): 64.6%
Elo Brier score:            0.220
Coin flip Brier score:      0.250


,elo_correct
season,
2019,0.640
2020,0.654
2021,0.604
2022,0.637
2023,0.614
2024,0.677
2025,0.646
2026,0.625


In [10]:
pbp_cols = ["game_id", "season", "week", "posteam", "defteam",
            "play_type", "epa", "success", "yards_gained"]

pbp = nfl.load_pbp(seasons).select(pbp_cols).to_pandas()
print("Plays loaded:", len(pbp))

print(pbp["play_type"].value_counts(dropna=False))

pbp = pbp[pbp["play_type"].isin(["pass", "run"]) & pbp["epa"].notna()]
print("\nPass and run plays kept:", len(pbp))

pbp[["epa", "success", "yards_gained"]].describe().round(3)

Plays loaded: 1106656
play_type
pass           454248
run            332292
no_play        103179
kickoff         63305
punt            56229
None            32879
extra_point     29305
field_goal      24075
qb_kneel         9331
qb_spike         1813
Name: count, dtype: int64

Pass and run plays kept: 786536


,epa,success,yards_gained
count,786536.000,786536.000,786536.000
mean,-0.009,0.428,5.431
std,1.383,0.495,8.674
min,-13.585,0.000,-38.000
25%,-0.623,0.000,0.000
50%,-0.189,0.000,3.000
75%,0.689,1.000,8.000
max,8.928,1.000,99.000


In [11]:
done_ids = set(schedules.loc[(schedules["season"] == 2026) & schedules["result"].notna(), "game_id"])
pbp_ids = set(pbp.loc[pbp["season"] == 2026, "game_id"])
missing = sorted(done_ids - pbp_ids)

print(f"2026 games with final scores: {len(done_ids)}")
print(f"2026 games with play by play: {len(done_ids & pbp_ids)}")
if missing:
    print("WARNING, play by play not posted yet for:", missing)
    print("Wait a few hours and run again.")
else:
    print("Data is complete. Safe to continue.")

2026 games with final scores: 64
2026 games with play by play: 63
WARNING, play by play not posted yet for: ['2026_04_ATL_NO']
Wait a few hours and run again.


In [12]:
offense = pbp.groupby(["game_id", "posteam"]).agg(
    off_epa=("epa", "mean"),
    off_success=("success", "mean"),
    off_ypp=("yards_gained", "mean"),
    off_plays=("epa", "size"),
).reset_index().rename(columns={"posteam": "team"})

defense = pbp.groupby(["game_id", "defteam"]).agg(
    def_epa=("epa", "mean"),
    def_success=("success", "mean"),
    def_ypp=("yards_gained", "mean"),
).reset_index().rename(columns={"defteam": "team"})

team_games = (team_games
    .merge(offense, on=["game_id", "team"], how="left")
    .merge(defense, on=["game_id", "team"], how="left"))

team_games["net_epa"] = team_games["off_epa"] - team_games["def_epa"]
team_games["net_success"] = team_games["off_success"] - team_games["def_success"]
team_games["net_ypp"] = team_games["off_ypp"] - team_games["def_ypp"]

print(team_games.shape)
print("Rows missing EPA:", team_games["off_epa"].isna().sum())

team_games[team_games["team"] == "KC"][
    ["game_id", "point_diff", "off_epa", "def_epa", "net_epa"]
].tail(5).round(3)

(12592, 25)
Rows missing EPA: 2


,game_id,point_diff,off_epa,def_epa,net_epa
6279,2025_18_KC_LV,-2.0,-0.468,-0.490,0.022
6280,2026_01_DEN_KC,21.0,0.164,-0.344,0.508
6281,2026_02_IND_KC,3.0,0.229,0.154,0.074
6282,2026_03_KC_MIA,14.0,0.138,-0.151,0.289
6283,2026_04_KC_LV,3.0,0.130,0.048,0.083


In [13]:
SPAN = 8

stats = ["point_diff", "points_for", "points_against", "win",
         "off_epa", "def_epa", "net_epa",
         "off_success", "def_success", "net_success",
         "off_ypp", "def_ypp", "net_ypp"]

team_games = team_games.sort_values(["team", "gameday"]).reset_index(drop=True)

for stat in stats:
    team_games[f"{stat}_ewm"] = (
        team_games.groupby("team")[stat]
        .transform(lambda s: s.shift(1).ewm(span=SPAN, min_periods=1).mean())
    )

compare = team_games[team_games["season"] >= 2005]

candidates = ["win_last8", "point_diff_last8",
              "win_ewm", "point_diff_ewm", "net_ypp_ewm",
              "net_success_ewm", "net_epa_ewm"]

print("How well each stat predicts the NEXT game's point differential:")
compare[candidates].corrwith(compare["point_diff"]).sort_values(ascending=False).round(3)

How well each stat predicts the NEXT game's point differential:


,0
net_epa_ewm,0.234
point_diff_ewm,0.234
point_diff_last8,0.232
win_ewm,0.212
net_ypp_ewm,0.209
win_last8,0.208
net_success_ewm,0.208


In [14]:
games = schedules.copy()
games["gameday"] = pd.to_datetime(games["gameday"])
games = games.sort_values(["gameday", "game_id"]).reset_index(drop=True)

games["rest_diff"] = games["home_rest"] - games["away_rest"]
games["neutral"] = (games["location"] == "Neutral").astype(int)

TIME_ZONE = {
    "NE": 0, "NYJ": 0, "NYG": 0, "BUF": 0, "MIA": 0, "PHI": 0, "PIT": 0,
    "BAL": 0, "WAS": 0, "CLE": 0, "CIN": 0, "DET": 0, "IND": 0, "ATL": 0,
    "CAR": 0, "JAX": 0, "TB": 0,
    "CHI": -1, "GB": -1, "MIN": -1, "KC": -1, "HOU": -1, "DAL": -1,
    "NO": -1, "TEN": -1,
    "DEN": -2,
    "ARI": -3, "LV": -3, "LAC": -3, "LA": -3, "SF": -3, "SEA": -3,
}
games["travel_hours"] = (games["home_team"].map(TIME_ZONE)
                         - games["away_team"].map(TIME_ZONE)).abs()

qb = pd.concat([
    games[["game_id", "gameday", "home_team", "home_qb_id"]]
        .rename(columns={"home_team": "team", "home_qb_id": "qb_id"}),
    games[["game_id", "gameday", "away_team", "away_qb_id"]]
        .rename(columns={"away_team": "team", "away_qb_id": "qb_id"}),
]).sort_values(["team", "gameday"])

qb["prev_qb"] = qb.groupby("team")["qb_id"].shift(1)
qb["qb_changed"] = ((qb["qb_id"] != qb["prev_qb"])
                    & qb["qb_id"].notna()
                    & qb["prev_qb"].notna()).astype(int)

for side in ["home", "away"]:
    games = games.merge(
        qb[["game_id", "team", "qb_changed"]].rename(
            columns={"team": f"{side}_team", "qb_changed": f"{side}_qb_changed"}),
        on=["game_id", f"{side}_team"], how="left")

print(games.shape)
games[["rest_diff", "neutral", "div_game", "travel_hours",
       "home_qb_changed", "away_qb_changed"]].mean().round(3)

(6504, 51)


,0
rest_diff,0.018
neutral,0.015
div_game,0.359
travel_hours,0.981
home_qb_changed,0.114
away_qb_changed,0.107


In [15]:
FORM = ["point_diff_ewm", "off_epa_ewm", "def_epa_ewm", "net_epa_ewm",
        "off_success_ewm", "def_success_ewm", "net_success_ewm", "net_ypp_ewm"]

past_form = team_games[["game_id", "team"] + FORM]

now_form = team_games[["team"]].copy()
for col in FORM:
    stat = col.replace("_ewm", "")
    now_form[col] = team_games.groupby("team")[stat].transform(
        lambda s: s.ewm(span=SPAN, min_periods=1).mean())
now_form = now_form.groupby("team").tail(1)

def attach_form(df, form, side, keys):
    renamed = form.rename(columns={"team": f"{side}_team",
                                   **{c: f"{side}_{c}" for c in FORM}})
    return df.merge(renamed, on=keys + [f"{side}_team"], how="left")

past = games[games["result"].notna()].merge(
    played[["game_id", "home_elo", "away_elo"]], on="game_id", how="left")

future = games[games["result"].isna()].copy()
future["home_elo"] = future["home_team"].map(elo)
future["away_elo"] = future["away_team"].map(elo)

for side in ["home", "away"]:
    past = attach_form(past, past_form, side, ["game_id"])
    future = attach_form(future, now_form, side, [])

model_df = pd.concat([past, future], ignore_index=True)
model_df["home_win"] = np.where(model_df["result"].isna(), np.nan,
                                (model_df["result"] > 0).astype(float))

model_df["elo_diff"] = model_df["home_elo"] - model_df["away_elo"]
for col in FORM:
    model_df[f"{col}_diff"] = model_df[f"home_{col}"] - model_df[f"away_{col}"]
model_df["qb_changed_diff"] = model_df["home_qb_changed"] - model_df["away_qb_changed"]

FEATURES = (["elo_diff"] + [f"{c}_diff" for c in FORM]
            + ["rest_diff", "neutral", "div_game", "travel_hours", "qb_changed_diff"])

print(model_df.shape)
print("Played:", model_df["home_win"].notna().sum(),
      "| Upcoming:", model_df["home_win"].isna().sum())
print("Features:", len(FEATURES))
print("Missing values from 2005 on:",
      model_df.loc[model_df["season"] >= 2005, FEATURES].isna().sum().sum())

(6504, 80)
Played: 6296 | Upcoming: 208
Features: 14
Missing values from 2005 on: 0


In [16]:
data = model_df[model_df["season"] >= 2005]
labeled = data[data["home_win"].notna()]

train = labeled[labeled["season"] <= 2021]
valid = labeled[labeled["season"].between(2022, 2023)]
test  = labeled[labeled["season"].between(2024, 2025)]
live  = labeled[labeled["season"] == 2026]
to_predict = model_df[model_df["home_win"].isna()]

for name, df in [("Train", train), ("Validation", valid), ("Test", test),
                 ("Live 2026", live), ("To predict", to_predict)]:
    rate = f"{df['home_win'].mean():.1%}" if df["home_win"].notna().any() else "n/a"
    print(f"{name:<11} {df['season'].min()}-{df['season'].max()}  "
          f"{len(df):>5} games   home win rate {rate}")

X_train, y_train = train[FEATURES], train["home_win"]
X_valid, y_valid = valid[FEATURES], valid["home_win"]
X_test,  y_test  = test[FEATURES],  test["home_win"]

Train       2005-2021   4559 games   home win rate 56.0%
Validation  2022-2023    569 games   home win rate 56.4%
Test        2024-2025    570 games   home win rate 54.0%
Live 2026   2026-2026     64 games   home win rate 56.2%
To predict  2026-2026    208 games   home win rate n/a


In [17]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, brier_score_loss

logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
logreg.fit(X_train, y_train)

valid_prob = logreg.predict_proba(X_valid)[:, 1]

elo_valid = 1 / (1 + 10 ** (-(valid["elo_diff"] + 48 * (1 - valid["neutral"])) / 400))

print(f"Logistic regression  accuracy {accuracy_score(y_valid, valid_prob > 0.5):.1%}"
      f"   Brier {brier_score_loss(y_valid, valid_prob):.4f}")
print(f"Elo alone            accuracy {accuracy_score(y_valid, elo_valid > 0.5):.1%}"
      f"   Brier {brier_score_loss(y_valid, elo_valid):.4f}")

weights = pd.Series(logreg[-1].coef_[0], index=FEATURES)
weights.sort_values(key=abs, ascending=False).round(3)

Logistic regression  accuracy 63.8%   Brier 0.2253
Elo alone            accuracy 62.6%   Brier 0.2275


,0
elo_diff,0.508
net_ypp_ewm_diff,0.217
point_diff_ewm_diff,0.143
qb_changed_diff,-0.133
def_epa_ewm_diff,0.105
rest_diff,0.077
neutral,-0.073
div_game,-0.071
net_epa_ewm_diff,-0.067
net_success_ewm_diff,0.053


In [18]:
import numpy as np

SITUATION = ["rest_diff", "neutral", "div_game", "travel_hours", "qb_changed_diff"]

feature_sets = {
    "Elo only":                     ["elo_diff"],
    "Elo + situation":              ["elo_diff"] + SITUATION,
    "Elo + situation + net EPA":    ["elo_diff", "net_epa_ewm_diff"] + SITUATION,
    "Elo + situation + off/def EPA": ["elo_diff", "off_epa_ewm_diff", "def_epa_ewm_diff"] + SITUATION,
    "Elo + situation + EPA + YPP":  ["elo_diff", "net_epa_ewm_diff", "net_ypp_ewm_diff"] + SITUATION,
    "All 14":                       FEATURES,
}

def walk_forward(cols, first=2014, last=2023):
    probs, actual = [], []
    for year in range(first, last + 1):
        train_part = labeled[labeled["season"] < year]
        valid_part = labeled[labeled["season"] == year]
        model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
        model.fit(train_part[cols], train_part["home_win"])
        probs.extend(model.predict_proba(valid_part[cols])[:, 1])
        actual.extend(valid_part["home_win"])
    probs, actual = np.array(probs), np.array(actual)
    return accuracy_score(actual, probs > 0.5), brier_score_loss(actual, probs)

for name, cols in feature_sets.items():
    acc, brier = walk_forward(cols)
    print(f"{name:<32} {len(cols):>2} features   accuracy {acc:.1%}   Brier {brier:.4f}")

Elo only                          1 features   accuracy 64.4%   Brier 0.2221
Elo + situation                   6 features   accuracy 64.7%   Brier 0.2215
Elo + situation + net EPA         7 features   accuracy 64.7%   Brier 0.2212
Elo + situation + off/def EPA     8 features   accuracy 64.4%   Brier 0.2209
Elo + situation + EPA + YPP       8 features   accuracy 64.6%   Brier 0.2213
All 14                           14 features   accuracy 64.9%   Brier 0.2211


In [19]:
from xgboost import XGBClassifier

FINAL_FEATURES = feature_sets["Elo + situation + off/def EPA"]
print("Final features:", FINAL_FEATURES)
print()

def walk_forward_model(make_model, cols, first=2014, last=2023):
    probs, actual = [], []
    for year in range(first, last + 1):
        train_part = labeled[labeled["season"] < year]
        valid_part = labeled[labeled["season"] == year]
        model = make_model()
        model.fit(train_part[cols], train_part["home_win"])
        probs.extend(model.predict_proba(valid_part[cols])[:, 1])
        actual.extend(valid_part["home_win"])
    probs, actual = np.array(probs), np.array(actual)
    return accuracy_score(actual, probs > 0.5), brier_score_loss(actual, probs)

candidates = {
    "Logistic regression": lambda: make_pipeline(StandardScaler(),
                                                 LogisticRegression(max_iter=1000)),
    "XGBoost (default)":   lambda: XGBClassifier(eval_metric="logloss"),
    "XGBoost (tuned down)": lambda: XGBClassifier(
        n_estimators=300, max_depth=2, learning_rate=0.02,
        subsample=0.8, colsample_bytree=0.8, eval_metric="logloss"),
}

for name, make_model in candidates.items():
    acc, brier = walk_forward_model(make_model, FINAL_FEATURES)
    print(f"{name:<24} accuracy {acc:.1%}   Brier {brier:.4f}")

Final features: ['elo_diff', 'off_epa_ewm_diff', 'def_epa_ewm_diff', 'rest_diff', 'neutral', 'div_game', 'travel_hours', 'qb_changed_diff']

Logistic regression      accuracy 64.4%   Brier 0.2209
XGBoost (default)        accuracy 59.1%   Brier 0.2567
XGBoost (tuned down)     accuracy 64.2%   Brier 0.2217


In [20]:
dev = labeled[labeled["season"] <= 2023]

exam_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
exam_model.fit(dev[FINAL_FEATURES], dev["home_win"])

test = test.copy()
test["model_prob"] = exam_model.predict_proba(test[FINAL_FEATURES])[:, 1]
test["elo_prob"] = 1 / (1 + 10 ** (-(test["elo_diff"] + 48 * (1 - test["neutral"])) / 400))

def implied_prob(moneyline):
    return np.where(moneyline < 0, -moneyline / (-moneyline + 100), 100 / (moneyline + 100))

home_raw = implied_prob(test["home_moneyline"])
away_raw = implied_prob(test["away_moneyline"])
test["vegas_prob"] = home_raw / (home_raw + away_raw)

y = test["home_win"]
contenders = {
    "Always pick home": np.full(len(test), 0.56),
    "Elo alone":        test["elo_prob"],
    "My model":       test["model_prob"],
    "Vegas":            test["vegas_prob"],
}
for name, p in contenders.items():
    print(f"{name:<18} accuracy {accuracy_score(y, np.asarray(p) > 0.5):.1%}"
          f"   Brier {brier_score_loss(y, p):.4f}")

test["bucket"] = pd.cut(test["model_prob"], [0, .3, .4, .5, .6, .7, 1])
test.groupby("bucket", observed=True).agg(
    games=("home_win", "size"),
    predicted=("model_prob", "mean"),
    actual=("home_win", "mean"),
).round(3)

Always pick home   accuracy 54.0%   Brier 0.2488
Elo alone          accuracy 65.6%   Brier 0.2168
My model           accuracy 65.1%   Brier 0.2160
Vegas              accuracy 68.4%   Brier 0.2057


,games,predicted,actual
bucket,,,
"(0.0, 0.3]",45,0.235,0.311
"(0.3, 0.4]",69,0.352,0.261
"(0.4, 0.5]",107,0.452,0.439
"(0.5, 0.6]",108,0.550,0.509
"(0.6, 0.7]",101,0.646,0.634
"(0.7, 1.0]",140,0.791,0.786


In [21]:
final_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
final_model.fit(labeled[FINAL_FEATURES], labeled["home_win"])

last_week = labeled.loc[labeled["season"] == 2026, "week"].max()
print(f"Trained on {len(labeled)} games, 2005 through 2026 Week {last_week}")

preds = to_predict[["game_id", "week", "gameday", "away_team", "home_team", "neutral"]].copy()
preds["home_win_prob"] = final_model.predict_proba(to_predict[FINAL_FEATURES])[:, 1]
preds["pick"] = np.where(preds["home_win_prob"] >= 0.5, preds["home_team"], preds["away_team"])
preds["confidence"] = np.maximum(preds["home_win_prob"], 1 - preds["home_win_prob"])
preds["gameday"] = preds["gameday"].dt.date
preds = preds.sort_values(["gameday", "game_id"]).reset_index(drop=True)

print("Games predicted:", len(preds))
preds[preds["week"] == 4][["gameday", "away_team", "home_team",
                           "home_win_prob", "pick", "confidence"]].round(3)

Trained on 5762 games, 2005 through 2026 Week 4
Games predicted: 208


,gameday,away_team,home_team,home_win_prob,pick,confidence


In [26]:
qb_cols = ["game_id", "passer_player_id", "rusher_player_id", "qb_dropback", "qb_epa"]
qb_plays = nfl.load_pbp(seasons).select(qb_cols).to_pandas()
qb_plays = qb_plays[(qb_plays["qb_dropback"] == 1) & qb_plays["qb_epa"].notna()].copy()
qb_plays["qb_id"] = qb_plays["passer_player_id"].fillna(qb_plays["rusher_player_id"])

qb_games = (qb_plays.groupby(["game_id", "qb_id"])
            .agg(epa_sum=("qb_epa", "sum"), dropbacks=("qb_epa", "size"))
            .reset_index()
            .merge(games[["game_id", "gameday"]], on="game_id")
            .sort_values(["qb_id", "gameday"]))

PRIOR, PHANTOM, WINDOW = -0.10, 100, 16

for c in ["epa_sum", "dropbacks"]:
    qb_games[f"{c}_before"] = (qb_games.groupby("qb_id")[c]
        .transform(lambda s: s.shift(1).rolling(WINDOW, min_periods=1).sum())
        .fillna(0))
    qb_games[f"{c}_after"] = (qb_games.groupby("qb_id")[c]
        .transform(lambda s: s.rolling(WINDOW, min_periods=1).sum()))

qb_games["rating_before"] = ((qb_games["epa_sum_before"] + PRIOR * PHANTOM)
                             / (qb_games["dropbacks_before"] + PHANTOM))
qb_games["rating_after"] = ((qb_games["epa_sum_after"] + PRIOR * PHANTOM)
                            / (qb_games["dropbacks_after"] + PHANTOM))

current_rating = qb_games.groupby("qb_id")["rating_after"].last()

done = games[games["result"].notna()]
starters = pd.concat([
    done[["gameday", "home_team", "home_qb_id"]].rename(columns={"home_team": "team", "home_qb_id": "qb_id"}),
    done[["gameday", "away_team", "away_qb_id"]].rename(columns={"away_team": "team", "away_qb_id": "qb_id"}),
]).dropna().sort_values("gameday")
latest_starter = starters.groupby("team")["qb_id"].last()

# Each QB's rating comes from his last game BEFORE this date,
# so a game's own play by play is never needed to rate it.
is_future = model_df["home_win"].isna()
qb_history = qb_games[["qb_id", "gameday", "rating_after"]].sort_values("gameday")
for side in ["home", "away"]:
    model_df = model_df.drop(columns=[f"{side}_qb_rating"], errors="ignore")
    lookup = (model_df[["game_id", "gameday", f"{side}_qb_id"]]
              .rename(columns={f"{side}_qb_id": "qb_id"})
              .sort_values("gameday"))
    matched = pd.merge_asof(lookup, qb_history, on="gameday", by="qb_id",
                            allow_exact_matches=False)
    model_df[f"{side}_qb_rating"] = model_df["game_id"].map(matched.set_index("game_id")["rating_after"])
    future_rating = model_df[f"{side}_team"].map(latest_starter).map(current_rating)
    model_df.loc[is_future, f"{side}_qb_rating"] = future_rating[is_future]
    model_df[f"{side}_qb_rating"] = model_df[f"{side}_qb_rating"].fillna(PRIOR)

model_df["qb_rating_diff"] = model_df["home_qb_rating"] - model_df["away_qb_rating"]

labeled = model_df[(model_df["season"] >= 2005) & model_df["home_win"].notna()]
to_predict = model_df[model_df["home_win"].isna()]

qb_names = pd.concat([
    games[["home_qb_id", "home_qb_name"]].set_axis(["qb_id", "name"], axis=1),
    games[["away_qb_id", "away_qb_name"]].set_axis(["qb_id", "name"], axis=1),
]).dropna().drop_duplicates("qb_id").set_index("qb_id")["name"]

qb_now = pd.DataFrame({"qb": latest_starter.map(qb_names),
                       "rating": latest_starter.map(current_rating)})
print(qb_now.sort_values("rating", ascending=False).round(3).head(5))
print(qb_now.sort_values("rating", ascending=False).round(3).tail(5))
print()

for label, cols in [("Without QB rating", FINAL_FEATURES),
                    ("With QB rating", FINAL_FEATURES + ["qb_rating_diff"])]:
    acc, brier = walk_forward(cols)
    print(f"{label:<18} accuracy {acc:.1%}   Brier {brier:.4f}")

                    qb  rating
team                          
SF         Brock Purdy   0.212
DAL       Dak Prescott   0.162
LA    Matthew Stafford   0.155
GB         Jordan Love   0.149
BUF         Josh Allen   0.149
                  qb  rating
team                        
CLE   Deshaun Watson  -0.065
TB     Jalon Daniels  -0.094
NYJ       Geno Smith  -0.099
TEN         Cam Ward  -0.113
CHI      Case Keenum  -0.141

Without QB rating  accuracy 65.3%   Brier 0.2185
With QB rating     accuracy 65.3%   Brier 0.2185


In [27]:
import os
os.makedirs("nfl_app", exist_ok=True)

FINAL_FEATURES = ["elo_diff", "def_epa_ewm_diff", "rest_diff", "neutral", "div_game",
                  "travel_hours", "qb_changed_diff", "qb_rating_diff"]
SHRINK_START, SHRINK_PER_WEEK = 0.95, 0.009

def implied_prob(moneyline):
    return np.where(moneyline < 0, -moneyline / (-moneyline + 100), 100 / (moneyline + 100))

def vegas_prob(df):
    home_raw = implied_prob(df["home_moneyline"])
    away_raw = implied_prob(df["away_moneyline"])
    return home_raw / (home_raw + away_raw)

def new_model():
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

# 1. Final model and the rest of 2026
final_model = new_model().fit(labeled[FINAL_FEATURES], labeled["home_win"])
last_week = int(labeled.loc[labeled["season"] == 2026, "week"].max())

future = to_predict[to_predict["game_type"] == "REG"].copy()
raw_prob = final_model.predict_proba(future[FINAL_FEATURES])[:, 1]
weeks_ahead = future["week"] - last_week
shrink = np.clip(SHRINK_START - SHRINK_PER_WEEK * weeks_ahead, 0, 1)
future["home_win_prob"] = 0.5 + shrink * (raw_prob - 0.5)
future["status"] = "Upcoming"

# 2. 2026 games already played, each predicted by a model that only saw earlier games
played_2026 = []
for wk in range(1, last_week + 1):
    before = labeled[(labeled["season"] < 2026) |
                     ((labeled["season"] == 2026) & (labeled["week"] < wk))]
    this_week = labeled[(labeled["season"] == 2026) & (labeled["week"] == wk)].copy()
    this_week["home_win_prob"] = new_model().fit(
        before[FINAL_FEATURES], before["home_win"]).predict_proba(this_week[FINAL_FEATURES])[:, 1]
    played_2026.append(this_week)
played_2026 = pd.concat(played_2026)
played_2026["status"] = "Final"

# 3. One table for the whole 2026 regular season
season_2026 = pd.concat([played_2026, future], ignore_index=True)
season_2026["vegas_prob"] = vegas_prob(season_2026)
season_2026["pick"] = np.where(season_2026["home_win_prob"] >= 0.5,
                               season_2026["home_team"], season_2026["away_team"])
season_2026["confidence"] = np.maximum(season_2026["home_win_prob"], 1 - season_2026["home_win_prob"])
season_2026["winner"] = np.where(season_2026["status"] == "Upcoming", None,
                          np.where(season_2026["result"] > 0, season_2026["home_team"],
                          np.where(season_2026["result"] < 0, season_2026["away_team"], "TIE")))
season_2026["correct"] = np.where(season_2026["status"] == "Upcoming", np.nan,
                                  (season_2026["pick"] == season_2026["winner"]).astype(float))
season_2026["gameday"] = pd.to_datetime(season_2026["gameday"]).dt.date
season_2026 = season_2026.sort_values(["gameday", "game_id"])
season_2026[["game_id", "week", "gameday", "away_team", "home_team", "away_score", "home_score",
             "status", "home_win_prob", "vegas_prob", "pick", "confidence", "winner", "correct",
             "neutral", "stadium"]].to_csv("nfl_app/games_2026.csv", index=False)

# 4. Team table: ratings now, record so far, projected wins
teams = pd.DataFrame({"elo": pd.Series(elo)})
teams["qb"] = latest_starter.map(qb_names)
teams["qb_rating"] = latest_starter.map(current_rating)
teams["def_epa"] = now_form.set_index("team")["def_epa_ewm"]
done_2026 = season_2026[season_2026["status"] == "Final"]
teams["wins"] = done_2026["winner"].value_counts().reindex(teams.index, fill_value=0)
teams["games"] = pd.concat([done_2026["home_team"], done_2026["away_team"]]).value_counts().reindex(teams.index, fill_value=0)
teams["losses"] = teams["games"] - teams["wins"]
home_left = future.groupby("home_team")["home_win_prob"].sum()
away_left = (1 - future["home_win_prob"]).groupby(future["away_team"]).sum()
teams["projected_wins"] = teams["wins"] + home_left.reindex(teams.index, fill_value=0) \
                          + away_left.reindex(teams.index, fill_value=0)
teams = teams.sort_values("elo", ascending=False).rename_axis("team").reset_index()
teams.insert(0, "rank", range(1, len(teams) + 1))
teams.to_csv("nfl_app/teams.csv", index=False)

# 5. Elo history since 2016 for the trend chart
elo_history = pd.concat([
    played[["gameday", "season", "home_team", "home_elo"]].set_axis(["gameday", "season", "team", "elo"], axis=1),
    played[["gameday", "season", "away_team", "away_elo"]].set_axis(["gameday", "season", "team", "elo"], axis=1),
])
elo_history[elo_history["season"] >= 2016].sort_values("gameday").to_csv("nfl_app/elo_history.csv", index=False)

# 6. Report card: final exam on 2024 and 2025 with the new features
dev = labeled[labeled["season"] <= 2023]
exam = labeled[labeled["season"].between(2024, 2025)].copy()
exam["model_prob"] = new_model().fit(dev[FINAL_FEATURES], dev["home_win"]).predict_proba(exam[FINAL_FEATURES])[:, 1]
exam["elo_prob"] = 1 / (1 + 10 ** (-(exam["elo_diff"] + 48 * (1 - exam["neutral"])) / 400))
exam["vegas_prob"] = vegas_prob(exam)
exam["home_prob"] = 0.56

report = []
for label, col in [("Always pick home", "home_prob"), ("Elo alone", "elo_prob"),
                   ("My model", "model_prob"), ("Vegas", "vegas_prob")]:
    report.append({"method": label,
                   "accuracy": accuracy_score(exam["home_win"], exam[col] > 0.5),
                   "brier": brier_score_loss(exam["home_win"], exam[col])})
pd.DataFrame(report).to_csv("nfl_app/report.csv", index=False)
exam[["season", "week", "home_team", "away_team", "home_win", "model_prob"]].to_csv(
    "nfl_app/exam_2024_2025.csv", index=False)

print("Saved:", sorted(os.listdir("nfl_app")))
print(f"2026 so far: {int(done_2026['correct'].sum())} of {len(done_2026)} correct")
teams.head(10)

Saved: ['app.py', 'elo_history.csv', 'exam_2024_2025.csv', 'games_2026.csv', 'report.csv', 'teams.csv']
2026 so far: 38 of 64 correct


,rank,team,elo,qb,qb_rating,def_epa,wins,games,losses,projected_wins
0,1,SEA,1683.203084,Drew Lock,-0.005131,-0.156077,3,4,1,10.939977
1,2,SF,1634.453565,Brock Purdy,0.212306,0.058318,4,4,0,12.165071
2,3,BUF,1626.250351,Josh Allen,0.148519,0.056012,3,4,1,11.085620
3,4,MIN,1613.533403,Kyler Murray,0.041480,-0.209492,4,4,0,12.034061
4,5,JAX,1611.665004,Trevor Lawrence,0.106670,-0.071509,3,4,1,11.386424
5,6,NE,1589.580977,Drake Maye,0.103755,-0.045802,2,4,2,10.160243
6,7,LA,1585.376980,Matthew Stafford,0.155142,-0.090677,2,4,2,10.003443
7,8,DEN,1580.450871,Bo Nix,0.065395,0.004056,2,4,2,9.610062
8,9,CHI,1570.762727,Case Keenum,-0.141436,-0.045340,3,4,1,8.909805
9,10,KC,1566.639155,Patrick Mahomes,0.145246,-0.063217,4,4,0,11.388288


In [28]:
%%writefile nfl_app/app.py
import pandas as pd
import plotly.graph_objects as go
import streamlit as st

st.set_page_config(page_title="NFL 2026 Game Predictor", page_icon="🏈", layout="wide")

ACCENT = "#2a78d6"
MUTED = "#898781"
GRID = "rgba(137,135,129,0.25)"


@st.cache_data
def load():
    games = pd.read_csv("nfl_app/games_2026.csv", parse_dates=["gameday"])
    teams = pd.read_csv("nfl_app/teams.csv")
    elo_hist = pd.read_csv("nfl_app/elo_history.csv", parse_dates=["gameday"])
    report = pd.read_csv("nfl_app/report.csv")
    exam = pd.read_csv("nfl_app/exam_2024_2025.csv")
    return games, teams, elo_hist, report, exam


games, teams, elo_hist, report, exam = load()
final = games[games["status"] == "Final"]
upcoming = games[games["status"] == "Upcoming"]
last_week = int(final["week"].max()) if len(final) else 0
next_week = int(upcoming["week"].min()) if len(upcoming) else last_week


def style(fig, height=360):
    fig.update_layout(
        height=height, margin=dict(l=8, r=8, t=8, b=8), showlegend=False,
        hoverlabel=dict(font_size=13), bargap=0.35,
    )
    fig.update_xaxes(showgrid=False, linecolor=GRID)
    fig.update_yaxes(gridcolor=GRID, zeroline=False)
    return fig


def result_label(row):
    if row["status"] == "Upcoming":
        return ""
    return "✅ Correct" if row["correct"] == 1 else "❌ Missed"


# ---------- Header ----------
st.title("🏈 NFL 2026 Game Predictor")
st.caption(f"Logistic regression on nflverse data since 2003. Results through Week {last_week}. "
           "Rerun the notebook each week to refresh.")

model_row = report.set_index("method").loc["My model"]
vegas_right = ((final["vegas_prob"] >= 0.5) == (final["home_score"] > final["away_score"])).sum()

k1, k2, k3, k4 = st.columns(4)
k1.metric("2026 picks correct", f"{int(final['correct'].sum())} of {len(final)}  ({final['correct'].mean():.0%})")
k2.metric("Vegas favorite, 2026", f"{vegas_right} of {len(final)}  ({vegas_right / max(len(final), 1):.0%})")
k3.metric("Test accuracy, 2024 to 2025", f"{model_row['accuracy']:.1%}",
          help=f"Brier score {model_row['brier']:.3f} on 570 games the model never trained on")
k4.metric("Games left to predict", len(upcoming))

tab_week, tab_team, tab_power, tab_report = st.tabs(
    ["Weekly picks", "Team outlook", "Power rankings", "Model report card"])

# ---------- Weekly picks ----------
with tab_week:
    weeks = sorted(games["week"].unique())
    week = st.selectbox("Week", weeks, index=weeks.index(next_week), key="week")
    wk = games[games["week"] == week].copy()
    wk["Matchup"] = wk["away_team"] + " @ " + wk["home_team"] + wk["neutral"].map({1: " (neutral)", 0: ""})
    wk["Win chance"] = wk["confidence"] * 100
    wk["Vegas on pick"] = (wk["vegas_prob"].where(wk["pick"] == wk["home_team"], 1 - wk["vegas_prob"])) * 100
    wk["Score"] = wk.apply(lambda r: "" if r["status"] == "Upcoming"
                           else f"{r['away_team']} {int(r['away_score'])}, {r['home_team']} {int(r['home_score'])}", axis=1)
    wk["Result"] = wk.apply(result_label, axis=1)
    wk["Date"] = wk["gameday"].dt.strftime("%a %b %d")
    wk["Pick"] = wk["pick"]

    if (wk["status"] == "Final").any():
        done = wk[wk["status"] == "Final"]
        st.markdown(f"**Week {week}: {int(done['correct'].sum())} of {len(done)} correct**")
    else:
        st.markdown(f"**Week {week}: {len(wk)} games.** Picks further out are pulled toward 50% "
                    "because a lot can change before then.")

    st.dataframe(
        wk[["Date", "Matchup", "Pick", "Win chance", "Vegas on pick", "Score", "Result"]],
        hide_index=True, width="stretch",
        column_config={
            "Win chance": st.column_config.ProgressColumn("Win chance", min_value=50, max_value=100, format="%.0f%%"),
            "Vegas on pick": st.column_config.NumberColumn("Vegas on pick", format="%.0f%%"),
        },
    )

# ---------- Team outlook ----------
with tab_team:
    team_list = sorted(teams["team"])
    team = st.selectbox("Team", team_list, index=team_list.index(teams.iloc[0]["team"]), key="team")
    t = teams.set_index("team").loc[team]

    c1, c2, c3, c4 = st.columns(4)
    c1.metric("Record", f"{int(t['wins'])} and {int(t['losses'])}")
    c2.metric("Projected wins", f"{t['projected_wins']:.1f}")
    c3.metric(f"Power rank (Elo {t['elo']:.0f})", f"#{int(t['rank'])} of 32")
    c4.metric(f"Starting QB (rating {t['qb_rating']:+.3f})", t["qb"])

    sched = games[(games["home_team"] == team) | (games["away_team"] == team)].copy()
    is_home = sched["home_team"] == team
    sched["Opponent"] = sched["away_team"].where(is_home, sched["home_team"])
    sched["Where"] = is_home.map({True: "Home", False: "Away"}).where(sched["neutral"] == 0, "Neutral")
    sched["win_prob"] = sched["home_win_prob"].where(is_home, 1 - sched["home_win_prob"])
    sched["Result"] = sched.apply(
        lambda r: "" if r["status"] == "Upcoming" else ("W" if r["winner"] == team else ("T" if r["winner"] == "TIE" else "L")), axis=1)

    left, right = st.columns([3, 2])
    with left:
        st.markdown(f"**{team} win chance, game by game**")
        fig = go.Figure(go.Bar(
            x=sched["week"], y=sched["win_prob"] * 100,
            marker_color=[ACCENT if s == "Upcoming" else MUTED for s in sched["status"]],
            marker_cornerradius=4,
            customdata=sched[["Opponent", "Where", "Result"]],
            hovertemplate="<b>%{y:.0f}%</b> win chance<br>Week %{x} vs %{customdata[0]} (%{customdata[1]})"
                          "<br>%{customdata[2]}<extra></extra>",
        ))
        fig.add_hline(y=50, line_dash="dot", line_color=MUTED)
        fig.update_yaxes(range=[0, 100], ticksuffix="%")
        fig.update_xaxes(title="Week", dtick=1)
        st.plotly_chart(style(fig), width="stretch", theme="streamlit")
        st.caption("Gray bars are games already played, shown with the pregame win chance. Blue bars are upcoming. The dotted line is 50%.")
    with right:
        st.markdown("**Schedule**")
        st.dataframe(
            sched.assign(**{"Win chance": sched["win_prob"] * 100})[["week", "Opponent", "Where", "Win chance", "Result"]]
                 .rename(columns={"week": "Wk"}),
            hide_index=True, width="stretch", height=380,
            column_config={"Win chance": st.column_config.NumberColumn(format="%.0f%%")},
        )

    st.markdown(f"**{team} Elo rating since 2016**")
    h = elo_hist[elo_hist["team"] == team]
    fig = go.Figure(go.Scatter(
        x=h["gameday"], y=h["elo"], mode="lines", line=dict(color=ACCENT, width=2),
        hovertemplate="<b>%{y:.0f}</b><br>%{x|%b %d, %Y}<extra></extra>",
    ))
    fig.add_hline(y=1500, line_dash="dot", line_color=MUTED,
                  annotation_text="League average 1500", annotation_position="bottom left")
    st.plotly_chart(style(fig, 300), width="stretch", theme="streamlit")

# ---------- Power rankings ----------
with tab_power:
    st.markdown("**Projected 2026 wins** (wins so far plus the win chance of every game left)")
    ranked = teams.sort_values("projected_wins")
    fig = go.Figure(go.Bar(
        x=ranked["projected_wins"], y=ranked["team"], orientation="h",
        marker_color=ACCENT, marker_cornerradius=4,
        customdata=ranked[["wins", "losses", "qb"]],
        hovertemplate="<b>%{x:.1f}</b> projected wins<br>%{y}: %{customdata[0]} and %{customdata[1]} so far"
                      "<br>QB %{customdata[2]}<extra></extra>",
    ))
    fig.add_vline(x=8.5, line_dash="dot", line_color=MUTED)
    fig.update_xaxes(range=[0, 17], title="Projected wins", showgrid=True, gridcolor=GRID)
    fig.update_yaxes(showgrid=False)
    st.plotly_chart(style(fig, 720), width="stretch", theme="streamlit")
    st.caption("The dotted line at 8.5 wins is a .500 season.")

    st.markdown("**Full table**, sorted by Elo")
    table = teams.assign(Record=teams["wins"].astype(str) + " and " + teams["losses"].astype(str))
    st.dataframe(
        table[["rank", "team", "elo", "qb", "qb_rating", "def_epa", "Record", "projected_wins"]].rename(columns={
            "rank": "Rank", "team": "Team", "elo": "Elo", "qb": "QB", "qb_rating": "QB rating",
            "def_epa": "Defense EPA allowed", "projected_wins": "Projected wins"}),
        hide_index=True, width="stretch", height=600,
        column_config={
            "Elo": st.column_config.NumberColumn(format="%.0f"),
            "QB rating": st.column_config.NumberColumn(format="%+.3f"),
            "Defense EPA allowed": st.column_config.NumberColumn(format="%+.3f", help="Lower is better"),
            "Projected wins": st.column_config.ProgressColumn(min_value=0, max_value=17, format="%.1f"),
        },
    )

# ---------- Model report card ----------
with tab_report:
    left, right = st.columns(2)
    with left:
        st.markdown("**Final exam: 2024 and 2025 seasons (570 games)**")
        r = report.copy()
        fig = go.Figure(go.Bar(
            x=r["method"], y=r["accuracy"] * 100,
            marker_color=[ACCENT if m == "My model" else MUTED for m in r["method"]],
            marker_cornerradius=4,
            text=[f"{a:.1%}" for a in r["accuracy"]], textposition="outside",
            customdata=r["brier"],
            hovertemplate="<b>%{y:.1f}%</b> accuracy<br>%{x}<br>Brier %{customdata:.4f}<extra></extra>",
        ))
        fig.update_yaxes(range=[0, 80], ticksuffix="%", title="Games picked correctly")
        st.plotly_chart(style(fig), width="stretch", theme="streamlit")
        st.dataframe(r.assign(accuracy=r["accuracy"] * 100).rename(
            columns={"method": "Method", "accuracy": "Accuracy", "brier": "Brier (lower is better)"}),
            hide_index=True, width="stretch",
            column_config={"Accuracy": st.column_config.NumberColumn(format="%.1f%%"),
                           "Brier (lower is better)": st.column_config.NumberColumn(format="%.4f")})

    with right:
        st.markdown("**Calibration: when the model says X%, does it happen X% of the time?**")
        exam["bucket"] = pd.cut(exam["model_prob"], [0, .3, .4, .5, .6, .7, 1])
        cal = exam.groupby("bucket", observed=True).agg(
            predicted=("model_prob", "mean"), actual=("home_win", "mean"), games=("home_win", "size")).reset_index()
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=[0, 100], y=[0, 100], mode="lines", hoverinfo="skip",
                                 line=dict(color=MUTED, dash="dot", width=1)))
        fig.add_trace(go.Scatter(
            x=cal["predicted"] * 100, y=cal["actual"] * 100, mode="lines+markers",
            line=dict(color=ACCENT, width=2), marker=dict(size=10, color=ACCENT, line=dict(width=2, color="white")),
            customdata=cal["games"],
            hovertemplate="Model said <b>%{x:.0f}%</b><br>Home team won <b>%{y:.0f}%</b><br>%{customdata} games<extra></extra>",
        ))
        fig.update_xaxes(range=[0, 100], ticksuffix="%", title="Model's home win chance")
        fig.update_yaxes(range=[0, 100], ticksuffix="%", title="How often the home team won")
        st.plotly_chart(style(fig), width="stretch", theme="streamlit")
        st.caption("Dots near the dotted line mean the probabilities can be trusted.")

    st.markdown("**2026 week by week**")
    weekly = final.assign(vegas_right=(final["vegas_prob"] >= 0.5) == (final["home_score"] > final["away_score"]))
    weekly = weekly.groupby("week").agg(Games=("correct", "size"), Model=("correct", "sum"),
                                        Vegas=("vegas_right", "sum")).reset_index().rename(columns={"week": "Week"})
    st.dataframe(weekly.astype(int), hide_index=True, width="stretch")

Overwriting nfl_app/app.py


In [29]:
import subprocess, time, re

subprocess.run(["pkill", "-f", "streamlit"])
subprocess.run(["pkill", "-f", "cloudflared"])
time.sleep(2)

!wget -q -nc -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared

subprocess.Popen(["streamlit", "run", "nfl_app/app.py", "--server.headless", "true",
                  "--server.port", "8501", "--theme.primaryColor", "#2a78d6"],
                 stdout=open("streamlit.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)

subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
                 stdout=open("tunnel.log", "w"), stderr=subprocess.STDOUT)

url = None
for _ in range(30):
    time.sleep(1)
    found = re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", open("tunnel.log").read())
    if found:
        url = found[0]
        break

time.sleep(10)
print("dashboard:", url or "not ready yet, run this cell again")

dashboard: https://barnes-thesis-participation-thousand.trycloudflare.com
